# YouTube Video Automation & Upload Pipeline

Pipeline tu dong hoa:
1. Ghep cac file audio trong thu muc va tron nhac nen (neu co).
2. Chia audio thanh cac nhom (group) duoi 12 tieng neu tong do dai vuot qua gioi han.
3. Ghep video nen (lap lai) va chen intro vao dau moi tap (neu co).
4. Dang nhap YouTube OAuth 2.0 (ho tro che do OOB headless cho Colab).
5. Upload video len kenh YouTube va tu dong dua vao playlist.

## 1. Mount Google Drive (Tuy chon)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive")
except ImportError:
    print("Moi truong khong phai Google Colab, bo qua buoc nay.")

## 2. Cai dat FFmpeg va thu vien can thiet

In [ ]:
# Cai dat ffmpeg he thong va cac goi Python
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q -r requirements.txt

# Kiem tra ffmpeg & ffprobe
!ffmpeg -version | head -n 1
!ffprobe -version | head -n 1

## 3. Chuan bi OAuth Client Secrets
Upload file `client_secrets.json` tu may tinh neu chua co tren may ao.

In [ ]:
import os
try:
    from google.colab import files
    if not os.path.exists("client_secrets.json"):
        print("Chon file client_secrets.json tu may de upload:")
        files.upload()
    else:
        print("Da tim thay client_secrets.json")
except ImportError:
    print("Local environment: kiem tra file client_secrets.json trong thu muc goc.")

## 4. Thiet lap cau hinh Pipeline

In [ ]:
from core import Pipeline, PipelineConfig

config = PipelineConfig(
    name="TruyenDaiKy",
    title_pattern="Tap {index} - Truyen Dai Ky Hay Nhat",
    description="Theo doi tap {index} trong series.",
    playlist_url=None,                  # Co the dien URL playlist hoac ID
    tags="audiobook, truyen audio",
    category_id="22",                   # 22 = People & Blogs
    privacy_status="unlisted",          # public | private | unlisted
    start_index=1,
    audios_dir="./data/audios",          # Thu muc chua cac file am thanh
    video_file="./data/bg_video.mp4",    # Video nen lap lai
    info_video=None,                     # Video gioi thieu noi dau tap (tuy chon)
    background_music=None,               # Nhac nen (tuy chon)
    bg_music_volume="0.2",
    output_dir="./output",
    client_secrets_file="client_secrets.json",
    token_file="token.json",
    noauth_local_webserver=True,         # Dat True khi chay tren Colab/Server khong co man hinh
    delete_final_video_after_upload=False,
    keep_master_audio=False,
)

print("Da khoi tao config cho project:", config.name)

## 5. Chay Pipeline va Upload

In [ ]:
pipeline = Pipeline(config)
uploaded_ids = pipeline.run()
print("Hoan tat! Danh sach Video ID da upload:", uploaded_ids)